# TripPlanner AI Notebook

This notebook uses the same code as the Streamlit app to:
- query the Chroma RAG index and SQLite database
- use live tools for weather, visas, and travel advisories
- run an end-to-end travel query through the agent pipeline

## 1. Install Project Dependencies

In [3]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
conda-repo-cli 1.0.75 requires requests_mock, which is not installed.
tensorflow-intel 2.18.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.6 which is incompatible.
conda-repo-cli 1.0.75 requires clyent==1.2.1, but you have clyent 1.2.2 which is incompatible.
conda-repo-cli 1.0.75 requires PyYAML==6.0.1, but you have pyyaml 6.0.3 which is incompatible.
conda-repo-cli 1.0.75 requires requests==2.31.0, but you have requests 2.33.1 which is incompatible.


## 2. Load Project Modules and Model Config

This notebook expects to run from the repository root and reads the same `.env` values as the app.
It will use the live Groq model IDs from `config.py`.

In [14]:
from pathlib import Path
import json
import os
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / "config.py").exists():
    raise RuntimeError("Run this notebook from the Trip-Planner project root.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PROJECT_PYTHON = Path(r"C:\Program Files\Python312\python.exe")
if not PROJECT_PYTHON.exists():
    raise RuntimeError(f"Expected project Python at {PROJECT_PYTHON}")

from config import MODELS, GROQ_API_KEY, CHROMA_DB_PATH, RAG_COLLECTION_NAME, USER_HOME
from agent.pipeline import generate

MODEL_8B_ID = MODELS["Fast"]
MODEL_70B_ID = MODELS["Thinking"]
AVAILABLE_MODEL_IDS = [
    ("8B model", MODEL_8B_ID),
    ("70B model", MODEL_70B_ID),
]

if not GROQ_API_KEY:
    raise RuntimeError("GROQ_API_KEY is not set. Add it to .env or your environment before running this notebook.")


def run_project_json(code: str) -> dict:
    env = os.environ.copy()
    env.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
    env.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")
    completed = subprocess.run(
        [str(PROJECT_PYTHON), "-c", code],
        cwd=str(ROOT),
        text=True,
        capture_output=True,
        env=env,
    )
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr or completed.stdout or "Project helper process failed.")
    lines = [line for line in completed.stdout.splitlines() if line.strip()]
    if not lines:
        raise RuntimeError("Project helper process returned no stdout.")
    return json.loads(lines[-1])


print(f"Project root: {ROOT}")
print(f"Project execution python: {PROJECT_PYTHON}")
print(f"Default origin: {USER_HOME}")
print(f"Chroma path: {CHROMA_DB_PATH}")
print(f"RAG collection: {RAG_COLLECTION_NAME}")
print("Models:")
for label, model_id in AVAILABLE_MODEL_IDS:
    print(f"  {label}: {model_id}")

Project root: c:\Users\Sran\Desktop\Travel Planner\Trip-Planner
Project execution python: C:\Program Files\Python312\python.exe
Default origin: San Jose, CA
Chroma path: c:\Users\Sran\Desktop\Travel Planner\Trip-Planner\chroma_db
RAG collection: travel_knowledge
Models:
  8B model: llama-3.1-8b-instant
  70B model: llama-3.3-70b-versatile


## 3. Quick Model Connectivity Check

Use the shared `generate()` helper from the agent pipeline to confirm both model entries can answer a short prompt.

In [ ]:
probe_messages = [
    {"role": "system", "content": "You are a concise travel assistant."},
    {"role": "user", "content": "In one sentence, say what TripPlanner can help with."},
]

for label, model_id in AVAILABLE_MODEL_IDS:
    print(f"\n[{label}] {model_id}")
    reply = generate(
        model_id,
        probe_messages,
        max_new_tokens=80,
        temperature=0.2,
        use_cache=False,
    )
    print(reply)


[Fast] llama-3.1-8b-instant
TripPlanner is here to assist with planning and organizing your trips, including booking flights, hotels, and activities, as well as providing recommendations and itineraries tailored to your preferences.

[Thinking] llama-3.3-70b-versatile
TripPlanner can help with organizing and booking travel arrangements, including flights, hotels, and activities, to create a personalized and stress-free trip itinerary.


## 4. Inspect the RAG Index and SQLite Database

Check whether the Chroma index is populated and inspect the current SQLite table counts.

In [8]:
status = run_project_json('''
import json
import database
from rag.ingest import run_full_ingest
from rag.vector_store import collection_count

database.init_schema()
rag_docs = collection_count()
ingested_now = False
if rag_docs == 0:
    ingest_result = run_full_ingest()
    rag_docs = ingest_result.get("total", 0)
    ingested_now = True

print(json.dumps({
    "ingested_now": ingested_now,
    "rag_chunks": rag_docs,
    "sqlite": database.stats(),
}, ensure_ascii=False))
''')

print("RAG/DB status:")
pprint(status)

RAG/DB status:
{'ingested_now': False,
 'rag_chunks': 7737,
 'sqlite': {'destinations': 77,
            'destinations_by_kind': {'city': 48,
                                     'district': 13,
                                     'itinerary': 7,
                                     'park': 9},
            'listings': 2914,
            'listings_by_category': {'Buy': 189,
                                     'Do': 773,
                                     'Drink': 250,
                                     'Eat': 607,
                                     'See': 574,
                                     'Sleep': 521}}}


## 5. Run Direct RAG and Database Queries

This section probes both retrieval surfaces directly:
- Chroma semantic search for narrative context
- SQLite lookups for exact destination and listing rows

In [9]:
direct_results = run_project_json('''
import json
from agent.tools import semantic_search, search_destinations, search_listings

rag_hits = semantic_search(
    query="California park hiking scenic views",
    state="CA",
    kind="park",
    top_k=5,
)
destination_rows = search_destinations(state="CA", kind="park", limit=5)
listing_rows = search_listings(city="San Francisco", category="Eat", limit=5)

print(json.dumps({
    "rag_hits": rag_hits,
    "destinations": destination_rows[:3],
    "listings": listing_rows[:3],
}, ensure_ascii=False))
''')

print("Top RAG hits:")
for i, chunk in enumerate(direct_results["rag_hits"], 1):
    meta = chunk.get("metadata", {})
    print(f"{i}. {meta.get('city')} | {meta.get('kind')} | {meta.get('section')} | {chunk.get('source')} | score={chunk.get('score')}")
    print(chunk.get("text", "")[:220].replace("\n", " "))
    print("---")

print("\nStructured destination rows:")
pprint(direct_results["destinations"])

print("\nStructured listing rows:")
pprint(direct_results["listings"])

Top RAG hits:
1. Kings Canyon | park | See | parks/kings_canyon.txt#See:19 | score=0.7878
Driving in the parks provide mostly up-close views of trees, so the roadside vista points that do exist should not be overlooked: - The road to Cedar Grove provides many excellent views of the narrow Kings Canyon. - Betw
---
2. Sequoia | park | See | parks/sequoia.txt#See:19 | score=0.7878
Driving in the parks provide mostly up-close views of trees, so the roadside vista points that do exist should not be overlooked: - The road to Cedar Grove provides many excellent views of the narrow Kings Canyon. - Betw
---
3. Joshua Tree | park | Do | parks/joshua_tree.txt#Do:26 | score=0.7857
===Photography===  The odd shapes of the Joshua tree as well as the dramatic geology and desert scenery make the park a great place for photographers. As with many areas, photography is best in the early morning and late
---
4. Kings Canyon | park | Understand | parks/kings_canyon.txt#Understand:3 | score=0.7845
In the l

## 6. Call the Live Tools

These helpers fetch live data outside the local knowledge base for weather, visa guidance, and travel advisories.

In [10]:
from agent.tools import get_weather, get_visa_info, get_travel_advisory

weather = get_weather("Tokyo")
visa = get_visa_info("US", "Japan")
advisory = get_travel_advisory("Mexico")

print("Weather:")
pprint(weather)

print("\nVisa info:")
pprint(visa)

print("\nTravel advisory:")
pprint(advisory)

Weather:
{'city': 'Tokyo',
 'current': {'description': 'Partly cloudy',
             'feels_like_c': '25',
             'humidity_pct': '69',
             'temp_c': '22',
             'temp_f': '72',
             'wind_kmph': '29'},
 'fetched_at_utc': '2026-04-28T07:37Z',
 'forecast': [{'chance_of_rain_pct': '0',
               'date': '2026-04-28',
               'description': 'Sunny',
               'label': 'today',
               'max_c': '19',
               'max_f': '67',
               'min_c': '18',
               'min_f': '64',
               'wind_kmph': '19'},
              {'chance_of_rain_pct': '0',
               'date': '2026-04-29',
               'description': 'Partly Cloudy ',
               'label': 'tomorrow',
               'max_c': '19',
               'max_f': '67',
               'min_c': '18',
               'min_f': '65',
               'wind_kmph': '38'},
              {'chance_of_rain_pct': '100',
               'date': '2026-04-30',
               'descri

## 7. Build an End-to-End Notebook Agent

This helper uses the same router, tool execution, and grounded response flow as the app.

In [15]:
def ask_tripplanner(user_query: str, model_id: str, history: list[dict] | None = None) -> dict:
    history = history or []
    supported_model_ids = {candidate for _, candidate in AVAILABLE_MODEL_IDS}
    if model_id not in supported_model_ids:
        raise ValueError(f"model_id must be one of {sorted(supported_model_ids)}")

    payload = json.dumps(
        {"user_query": user_query, "model_id": model_id, "history": history},
        ensure_ascii=False,
    )

    code = f'''
import json
from agent.pipeline import route_query, execute_tools, generate_response, generate
from agent.prompts import RESPONSE_SYSTEM_PROMPT

payload = json.loads({payload!r})
user_query = payload["user_query"]
model_id = payload["model_id"]
history = payload["history"]
tool_calls = route_query(model_id, user_query, history=history)
tool_results, rag_chunks = execute_tools(tool_calls)

if not tool_results:
    messages = [{{"role": "system", "content": RESPONSE_SYSTEM_PROMPT}}]
    messages.extend(history[-6:])
    messages.append({{"role": "user", "content": user_query}})
    answer = generate(model_id, messages, max_new_tokens=500)
else:
    answer = generate_response(
        model_id,
        user_query,
        tool_results,
        rag_chunks=rag_chunks,
        history=history,
        stream=False,
    )

print(json.dumps({{
    "model_id": model_id,
    "user_query": user_query,
    "tool_calls": tool_calls,
    "tool_results": tool_results,
    "rag_chunks": rag_chunks,
    "answer": answer,
}}, ensure_ascii=False))
'''

    return run_project_json(code)

In [ ]:
demo_result = ask_tripplanner(
    "Compare Yosemite, Sequoia, and Joshua Tree for hiking, scenery, and trip style",
    model_id=MODEL_8B_ID,
)

print(f"Model: {MODEL_8B_ID}")
print("Tool calls:")
pprint(demo_result["tool_calls"])

print("\nAnswer:\n")
print(demo_result["answer"])

print("\nFirst 3 RAG chunks:")
for i, chunk in enumerate(demo_result["rag_chunks"][:3], 1):
    meta = chunk.get("metadata", {})
    print(f"{i}. {meta.get('city')} | {meta.get('section')} | {chunk.get('source')}")

## 8. Ask Your Own Travel Question

Edit the values below and rerun the next two cells to see the 8B and 70B outputs separately.

In [16]:
USER_QUERY = "Suggest a weekend trip from San Jose with coast views and good food"
CHAT_HISTORY = []

In [17]:
result_8b = ask_tripplanner(USER_QUERY, model_id=MODEL_8B_ID, history=CHAT_HISTORY)

print(f"Model: {MODEL_8B_ID}")
print("Tool calls:")
pprint(result_8b["tool_calls"])

print("\nAnswer:\n")
print(result_8b["answer"])

print("\nTop RAG chunks:")
for i, chunk in enumerate(result_8b["rag_chunks"][:5], 1):
    meta = chunk.get("metadata", {})
    print(f"{i}. {meta.get('city')} | {meta.get('section')} | {chunk.get('source')} | score={chunk.get('score')}")

Model: llama-3.1-8b-instant
Tool calls:
[{'args': {'query': 'weekend getaway from San Jose with coast views and good '
                    'food',
           'state': 'CA'},
  'tool': 'semantic_search'}]

Answer:

Based on your request for a weekend trip from San Jose with coast views and good food, I recommend a day trip to Santa Cruz, a small coastal city about an hour's drive from San Jose [6]. Spend the day enjoying the beaches and Boardwalk, or make it the first stop on a longer coastal drive.

In Santa Cruz, you can try the Mexican food at one of the many local restaurants, such as those mentioned in passage [5]. Alternatively, head to Monterey, which is a bit further south, and try some of the fresh seafood at one of the many waterfront restaurants in Fisherman's Wharf, such as the Buena Vista Cafe [7].

If you'd rather stay in San Jose, you can explore the city's downtown area and visit the San Pedro Square Farmer's Market on Fridays for local and organic produce [2]. You can a

In [18]:
result_70b = ask_tripplanner(USER_QUERY, model_id=MODEL_70B_ID, history=CHAT_HISTORY)

print(f"Model: {MODEL_70B_ID}")
print("Tool calls:")
pprint(result_70b["tool_calls"])

print("\nAnswer:\n")
print(result_70b["answer"])

print("\nTop RAG chunks:")
for i, chunk in enumerate(result_70b["rag_chunks"][:5], 1):
    meta = chunk.get("metadata", {})
    print(f"{i}. {meta.get('city')} | {meta.get('section')} | {chunk.get('source')} | score={chunk.get('score')}")

Model: llama-3.3-70b-versatile
Tool calls:
[{'args': {'query': 'weekend trip from San Jose with coast views and good food',
           'state': 'CA'},
  'tool': 'semantic_search'}]

Answer:

For a weekend trip from San Jose with coast views and good food, I suggest heading to Santa Cruz, which is less than an hour away over the scenic Santa Cruz Mountains [3]. You can enjoy the beaches and Boardwalk in Santa Cruz, and then take Route 1 (also known as the Pacific Coast Highway) south to Capitola, Monterey, and the charming town of Carmel-by-the-Sea [3]. 

As for food, you can try some of the great seafood options in the area. Although the passages don't specifically mention seafood restaurants in Santa Cruz or Monterey, they do mention Scott's Seafood Grill and Bar in San Jose, which has a classic selection of seafood, as well as pasta and steaks from the grill [8]. 

The knowledge base doesn't have specific info on coastal restaurants near Santa Cruz, but in general, you can find great